# Real World MCP: Augmenting Model Context from Existing Data Sources

Today we will use **Google MCP Toolbox** to connect agents to two local databases running in Docker. We will build an LLM-based agent with **OpenRouter**, wire up tool calling, and drive everything with **YAML tool definitions** so the agent can answer questions in a chat-like way.

After you run the notebook, you will practice your own queries and extend the toolbox with new tools.

### Quickstart

Run these steps in your terminal from the **repository root** unless noted otherwise.

1. **Python environment** — Create a venv and install the project in editable mode:

    ```bash
    python3.12 -m venv .venv

    # macOS / Linux:
    source .venv/bin/activate
    # Windows cmd:
    .venv\Scripts\activate.bat
    # Windows PowerShell:
    .venv\Scripts\Activate.ps1

    pip install -U pip
    pip install -e .
    ```

2. **Environment file** — Copy [`.env.example`](../.env.example) to `.env` in the repo root.

3. **OpenRouter** — Put your OpenRouter API key in the `OPENROUTER_API_KEY` section of `.env` (this is the main secret you must supply).

4. **Other Configurations** — Feel free to change other sections of `.env` such as the model you will be using.

4. **Databases** — Start Postgres, Mongo, and the seed job:

    ```bash
    docker compose up --build
    ```


**Troubleshooting:** See the main [README](../README.md) if anything fails.  

---

# I. Tutorial

### Dataset and Databases
**Our dataset is mirrored from [**126 Years of Historical Olympic Data**](https://www.kaggle.com/datasets/muhammadehsan02/126-years-of-historical-olympic-dataset?select=Olympic_Athlete_Event_Details.csv) on Kaggle. Running the docker-compose file creates and populates two databases:**

#### A Postgres (SQL) database with 4 tables:  
- **olympic_games** provides a summary of each Olympic Games edition, including key details such as the year, host city, and country.
- **athletes** is a slim table capturing athlete name, gender, and country of birth (as well as the primary key athlete_id referenced in other tables).
- **countries** provides a mapping of National Olympic Committee (NOC) codes to country names.
- **athlete_events** contains detailed results of Olympic events for each athlete from 1896 to 2022. Each row represents one athlete's participation in a single event; each event can have multiple participants, and each athlete can have participated in multiple events.

#### A MongoDB document database with 2 collections:
- **olympic_athlete_biography** provides detailed biographical information on each Olympic athlete represented in the dataset.
- **olympic_event_results** contains detailed results of each Olympic event represented in the dataset.

## 1. MCP Toolbox Configuration

**The docker-compose file also sets up our MCP server:**
- The Toolbox service wraps our databases in a local, unsecured MCP server.
- The tools our agent will use are configured at runtime via the YAML files in the `/toolbox/config/tools` folder.
- At the end of this notebook you will use these same files to define your own custom tools.

Advanced concepts such as security/auth and additional database configuration guidance can be found in the official [MCP Toolbox for Databases](https://mcp-toolbox.dev/documentation/introduction/) documentation.

### 1.1 **Sources**

Our first YAML file ([**01_sources.yaml**](../toolbox/config/tools/01_sources.yaml)) defines our **sources** - these are the named database connections against which we will run our queries. As we define our tools, we will point to specific sources using `source: <name>`. Below is the source for our Postgres database (notice that we are pulling most of these values from our environment variables).

```yaml
kind: source
name: olympics-postgres
type: postgres
host: ${POSTGRES_HOST}
port: ${POSTGRES_PORT}
database: ${POSTGRES_DB}
user: ${POSTGRES_USER}
password: ${POSTGRES_PASSWORD}
```

### 1.2 **Tools**

The other YAML documents in our toolbox directory define our **tools**, which our agent will use to interact with our databases. MCP Toolbox automatically merges every `*.yaml` under `toolbox/config/tools/`, so we can subdivide our tools into logical groups.

Below you can see `pg_search_countries` from [`02_postgres_tools.yaml`](../toolbox/config/tools/02_postgres_tools.yaml). This file defines all of the tools for our Postgres database, and this particular tool is designed to look up countries based on their NOC code.

We can see that this tool is identified by a unique `name`, runs a `statement` (in this case a SQL query) on the linked `source`, and identifies the parameters the model will fill in (mapped to `$1`, `$2`, ... in order).

```yaml
kind: tool
name: pg_search_countries
type: postgres-sql
source: olympics-postgres
description: >
  Search countries by name fragment and return NOC codes (National Olympic Committee).
  Use before medal tools when the user says e.g. "United States" instead of "USA".
parameters:
  - name: name_fragment
    type: string
    description: Substring to match against country name (case-insensitive).
statement: |
  SELECT noc, country
  FROM countries
  WHERE country ILIKE '%' || $1 || '%'
  ORDER BY country ASC
  LIMIT 50
annotations:
  readOnlyHint: true
  destructiveHint: false
```

### 1.5 **Toolsets**

Toolsets (defined for our MCP server in [**04_toolsets.yaml**](toolbox/config/tools/04_toolsets.yaml)) are named bundles of tools, which the MCP server can expose to agents as a group. 

In this case, we have declared three toolsets:
- **postgres_only** - only the Postgres SQL tools (pg_*).
- **mongo_only** — only the Mongo tools (mongo_*).
- **combined** — all of those Postgres and Mongo tools in one list.

MCP clients (like the one we are about to build) can use these toolsets to fetch subsets of tools that are relevant to specific tasks. This serves as an important means to control which agents have access to which tools, and under what circumstances. It also means that we do not have to include all of our tool definitions in every API call to the LLM powering our agent (which for a larger corpus of tools can greatly increase token usage and the cost of running an agent).

### 1.4 **Prompts**

MCP Toolbox also supports storing **prompts** in our yaml files. This is beyond the scope of this tutorial, but a useful way to organize the various system prompts you want your agents to use for specific tasks. We have a few examples [here](../toolbox/config/tools/05_prompts.yaml) to demonstrate the concept.

---

## 2. Environment and Settings

Before we build our agent, let's import our Python libraries and load our settings.

#### Key libraries

| Package | Role in this tutorial |
| :--- | :--- |
| **`python-dotenv`** (`dotenv`) | Loads `.env` so API keys and URLs (OpenRouter, Toolbox) are available without hard-coding secrets. |
| **`openai`** | Official OpenAI SDK in **OpenAI-compatible** mode: OpenRouter uses the same request/response shapes, so we use it for chat completions and tool/function calling with OpenRouter credentials. |
| **`toolbox-core`** | Google **MCP Toolbox** Python client: connects to the local Toolbox MCP server, discovers tools from our YAML files, and invokes them as `ToolboxSyncTool` over the MCP protocol. |

**Docs:** [OpenRouter quickstart](https://openrouter.ai/docs/quickstart) (keys and OpenAI-compatible base URL); [`toolbox-core` on PyPI](https://pypi.org/project/toolbox-core/); [Python SDK core guide](https://googleapis.github.io/genai-toolbox/sdks/python-sdk/core/) ([sync client](https://googleapis.github.io/genai-toolbox/sdks/python-sdk/core/#synchronous-usage)).


In [ ]:
import os
import json
import random
from typing import Any
from pathlib import Path
from dataclasses import dataclass
from collections.abc import Sequence

from openai import OpenAI
from dotenv import load_dotenv
from toolbox_core import ToolboxSyncClient
from toolbox_core.protocol import Protocol
from toolbox_core.sync_tool import ToolboxSyncTool

AGENT_MAX_TOOL_ROUNDS = 16


@dataclass(frozen=True)
class Settings:
    """URLs and keys from the environment; ``agent_max_tool_rounds`` is fixed here."""

    openrouter_api_key: str
    openrouter_base_url: str
    openrouter_model: str
    toolbox_base_url: str
    agent_max_tool_rounds: int

    @staticmethod
    def from_env() -> "Settings":
        key = (os.environ.get("OPENROUTER_API_KEY") or "").strip()
        base = (
            os.environ.get("OPENROUTER_BASE_URL") or "https://openrouter.ai/api/v1"
        ).strip()
        model = (os.environ.get("OPENROUTER_MODEL") or "openai/gpt-4o-mini").strip()
        toolbox = (
            (os.environ.get("TOOLBOX_BASE_URL") or "http://127.0.0.1:5050")
            .strip()
            .rstrip("/")
        )
        return Settings(
            openrouter_api_key=key,
            openrouter_base_url=base,
            openrouter_model=model,
            toolbox_base_url=toolbox,
            agent_max_tool_rounds=AGENT_MAX_TOOL_ROUNDS,
        )


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError("Run from inside the repo clone (need pyproject.toml).")


load_dotenv(repo_root() / ".env")
settings = Settings.from_env()


---

## 3. Tool-calling loop

Now that we understand the basic building blocks of our MCP server, let's go step-by-step and build a simple agent using a **tool-calling loop**.

### 3.1 JSON Schema type names and Toolbox-to-OpenAI helper function

In order to utilize our tool definitions from the YAML files used by MCP Toolbox, we need a helper to convert them to the static JSON format expected in an LLM call. In the cell below, we first create a mapping between the data types used by each format, and then we implement the mapper function.



In [ ]:
# ``json`` parses each tool's argument string from the model into a Python dict.
# ``Sequence`` / ``Any`` keep type hints readable for ``ToolboxSyncTool`` lists and message dicts.

# Toolbox names parameter kinds with short strings; OpenAI expects JSON Schema types here.
_JSON_TYPES = {
    "string": "string",
    "integer": "integer",
    "float": "number",
    "boolean": "boolean",
    "array": "array",
    "object": "object",
}

def toolbox_tools_to_openai(tools: Sequence[ToolboxSyncTool]) -> list[dict[str, Any]]:
    """Map toolbox_core tools to OpenAI chat ``tools=`` shape (happy path)."""
    # One dict per tool; the chat API exposes these as functions the model may invoke.
    out: list[dict[str, Any]] = []
    for t in tools:
        # JSON Schema object for this function's arguments (names, types, which are required).
        props: dict[str, Any] = {}
        required: list[str] = []
        # ``t._params`` is toolbox_core's internal list of parameter metadata for this tool.
        for p in t._params:
            props[p.name] = {
                "type": _JSON_TYPES.get(p.type, "string"),
                "description": p.description,
            }
            if p.required:
                required.append(p.name)
        out.append(
            {
                "type": "function",
                "function": {
                    # Must match the strings the model emits in ``tool_calls[].function.name``.
                    "name": t._name,
                    # Shown to the model only; helps it choose the right tool and arguments.
                    "description": (t._description or "").strip(),
                    "parameters": {
                        "type": "object",
                        "properties": props,
                        "required": required,
                        "additionalProperties": False,
                    },
                },
            }
        )
    return out

### 3.2 Look up tools by function name

The LLM powering our agent does not run our Python tools directly; it will return the strings **function.name** and **function.arguments** for each tool it wants to use. Our code must run the tool, and feed the result back to the LLM.

This means that our next step is to build a helper function that builds a dictionary mapping each **function.name** string to the corresponding Toolbox tool object that our code will run.


In [ ]:
def tool_by_name(tools: Sequence[ToolboxSyncTool]) -> dict[str, ToolboxSyncTool]:
    # When the model picks ``function.name``, we look up the live callable bound to Toolbox.
    return {t._name: t for t in tools}


### 3.3 OpenRouter via the OpenAI SDK

Next we create a function to instantiate an OpenAI client. Note that we are pointing base_url at OpenRouter, which means we are using the **OpenAI format**, but route calls through **OpenRouter's endpoint** to access our LLM.

In [ ]:
def openrouter_client(settings: Settings) -> OpenAI:
    # Same ``openai`` SDK as OpenAI; ``base_url`` + ``api_key`` point at OpenRouter instead.
    return OpenAI(
        api_key=settings.openrouter_api_key,
        base_url=settings.openrouter_base_url,
        default_headers={
            "HTTP-Referer": "https://github.com/Rotational-io/mcp-tutorial",
            "X-Title": "mcp-tutorial",
        },
    )


### 3.4 Build the first `messages` list

Each call to the chat API sends the whole conversation history as a list of messages. We build the first message with the usual two roles: system (behavior and constraints) and user (the actual question).

Later, when the model asks for tools, we will append an assistant turn (including the tool call ids) and one tool turn per result.

In [ ]:
def initial_messages(
    system_prompt: str,
    user_prompt: str,
    cap: int,
) -> list[dict[str, Any]]:
    # Keep the model's plan aligned with how many tool-bearing turns the client allows.
    sys_text = (
        system_prompt.rstrip()
        + f"\n\nAt most {cap} model turns may include tool calls."
    )
    # Chat Completions expects a time-ordered list; we will append assistant + tool rows next.
    return [
        {"role": "system", "content": sys_text},
        {"role": "user", "content": user_prompt},
    ]


### 3.5 Ask the model once

Next we wrap a single `chat.completions.create` call in a small helper. Each time we call it we send the full messages transcript built so far, and the same tools list (the Toolbox catalog we mapped to OpenAI’s shape). The model can answer in plain text, request tool calls, or both depending on the reply.

We set `tool_choice="auto"` so the API decides whether to invoke tools; we are not forcing a tool on every turn.


In [ ]:
def call_model(
    oai: OpenAI,
    settings: Settings,
    messages: list[dict[str, Any]],
    oai_tools: list[dict[str, Any]],
):
    """One ``chat.completions`` round (reply may include ``tool_calls``)."""
    return oai.chat.completions.create(
        model=settings.openrouter_model,
        messages=messages,  # full transcript so far (system, user, prior tools, ...)
        tools=oai_tools,  # same catalog every round for this Toolbox session
        tool_choice="auto",  # model may answer with text only, call tools, or mix
    )


### 3.6 Record tool calls and append tool results

When the model’s reply includes tool_calls, the chat API expects you to extend the transcript in a strict order before you call the model again:

1. Append an assistant message that replays those calls: same `id`, `function.name`, and `function.arguments` (still a JSON string) for each one. That row is the record of what the model asked for.  

2. Append one tool message per call. Each tool row must use the matching `tool_call_id` (same value as the `id` from step 1) and put the tool output in content as a string—here that string is whatever Toolbox returns from running the tool.

If either step is missing or the ids do not line up, the next completion request is invalid. The two helpers in the next cell adhere to that pattern: first mirror the assistant turn, then run each tool and append the results.


In [ ]:
def append_assistant_with_tool_calls(messages: list[dict[str, Any]], msg) -> None:
    """Append assistant turn including ``tool_calls`` (ids link to the next rows)."""
    # Without this assistant row (including ids), the API rejects the following ``tool`` rows.
    messages.append(
        {
            "role": "assistant",
            "content": msg.content,  # often empty when the model only wants tools
            "tool_calls": [
                {
                    "id": tc.id,  # must match ``tool_call_id`` on each tool message below
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,  # JSON object as a string
                    },
                }
                for tc in msg.tool_calls
            ],
        }
    )


def append_tool_outputs(
    messages: list[dict[str, Any]],
    msg,
    by_name: dict[str, ToolboxSyncTool],
) -> None:
    """Parse JSON arguments, invoke Toolbox tool, append ``role: tool`` rows."""
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments or "{}")
        # Blocking call into MCP Toolbox; ``payload`` is a string (often JSON) for the model.
        payload = by_name[tc.function.name](**args)
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tc.id,
                "content": payload,
            }
        )


### 3.7 Put it together: `chat_with_tools`

Now we are ready to put all of this together in a single `chat_with_tools` function. We open one MCP session to Toolbox for this user turn (so every tool call shares the same connection) then load the combined toolset (which we defined in `04_toolsets.yaml`). From that we use our helper functions to build the OpenAI-style tools list for the model and the name-to-tool map our code uses to execute each tool call selected by the model.

After that we have a simple loop. We call `call_model` with whatever messages we have so far. If the model comes back with `tool_calls` we append the assistant row, run the tools, append the tool results, and ask again with another `call_model` call. If it answers with plain text and no tool calls, we are done; that string is the reply. If we hit the round cap before we receive our reply, we exit the loop and return a “round limit” message instead.

Note that within this function we have also implemented a `verbose` flag that will allow us to toggle between printing only the agent's answer and seeing the whole list of tool calls and "conversation" the agent has while interacting with our databases.

In [ ]:
def chat_with_tools(
    settings: Settings,
    *,
    user_prompt: str,
    system_prompt: str,
    max_rounds: int | None = None,
    verbose: bool = False,
) -> str:
    """One user turn with Toolbox tools (happy path)."""
    cap = max_rounds if max_rounds is not None else settings.agent_max_tool_rounds
    oai = openrouter_client(settings)
    messages = initial_messages(system_prompt, user_prompt, cap)
    _tool_out_limit = 4000

    # Keep the client open for the whole user turn so every tool call shares one MCP session.
    with ToolboxSyncClient(
        settings.toolbox_base_url,
        protocol=Protocol.MCP_LATEST,
    ) as tb:
        loaded = tb.load_toolset("combined")  # name from toolbox/config/tools/04_toolsets.yaml
        by_name = tool_by_name(loaded)
        oai_tools = toolbox_tools_to_openai(loaded)

        for round_idx in range(cap):
            resp = call_model(oai, settings, messages, oai_tools)
            msg = resp.choices[0].message
            if verbose:
                print(f"\n--- model turn {round_idx + 1} ---", flush=True)
                if msg.content and str(msg.content).strip():
                    print("assistant (text):", str(msg.content).strip(), flush=True)
                if msg.tool_calls:
                    for tc in msg.tool_calls:
                        args = json.loads(tc.function.arguments or "{}")
                        print(f"tool_call id={tc.id} name={tc.function.name}", flush=True)
                        print(json.dumps(args, indent=2), flush=True)
            if msg.tool_calls:
                # Record what the model asked for, run tools, then ask again with a longer transcript.
                append_assistant_with_tool_calls(messages, msg)
                append_tool_outputs(messages, msg, by_name)
                if verbose:
                    for row in messages[-len(msg.tool_calls) :]:
                        body = row["content"]
                        if len(body) > _tool_out_limit:
                            body = body[:_tool_out_limit] + "\n… [truncated]"
                        print(f"tool_result id={row['tool_call_id']}", flush=True)
                        print(body, flush=True)
                continue
            # No ``tool_calls``: treat assistant text as the final answer for this user message.
            if verbose:
                print("\n--- final assistant reply ---", flush=True)
                print((msg.content or "").strip(), flush=True)
            return (msg.content or "").strip()

    # Only runs if ``cap`` rounds all ended with tool calls (never a text-only finish inside the loop).
    if verbose:
        print("\n--- round limit (no text-only finish) ---", flush=True)
    return "Round limit reached without a final text reply."


---

## 4. Examples
Now let's put our agent to work and use it to ask some questions about historical Olympic data:

### Example A — structured data then bios

Here we are going to ask our agent a question that allows it to make some choices. After giving it the persona of a "tutorial assistant with Olympic data tools" in the system prompt, we give it a range of years from which to select a specific Olympic Games, and a list of countries to choose from. We then ask it to return information from the biographies of some of the athletes from that country who medaled during those games.

This is an interesting query because it not only allows the agent to make some unique choices, but also requires it to combine information that is stored separately in the Postgres and Mongo databases. Try running the cell as-is, and review the results. Then uncomment the line with the `verbose` flag, and run it again to follow the tool calls and the "conversation" the agent has while interacting with our databases. Does its workflow match what you would expect?

In [ ]:
TUTORIAL_SYSTEM_PROMPT = (
    "You are a tutorial assistant with Olympic data tools (relational + document). "
    "Use tools for facts; do not invent athlete, edition, or country codes. "
    "Call tools yourself—do not ask the user to run queries. "
    "Keep tool use reasonable in depth (a few biography lookups per answer is enough). "
    "Prefer concise answers with citations to ids returned by tools."
)

PROMPT_A = (
    "Choose a Summer Olympic Games held between 1992 and 2020, and choose one of these "
    "countries to focus on: United States, Australia, Japan, France, or Kenya. Who won "
    "medals for that country at those Games (any sport)? Use the data tools, mention a "
    "few medalists with a line or two from their biographies when you can, and cite "
    "athlete ids from the tools."
)

answer_a = chat_with_tools(
    settings,
    user_prompt=PROMPT_A,
    system_prompt=TUTORIAL_SYSTEM_PROMPT,
    #verbose=True
)
print(answer_a)

### Example B — biography search then cross-check

Here we flip the story from Example A. We still use the same tutorial assistant setup, but the user prompt starts from a sport theme in plain language (this notebook picks one phrase at random from a small list). We expect this to nudge the model toward a text-oriented search over the athlete biographies with mongo_* tools first, then follow-up calls to retrieve structured information Postgres. The answer should compare what the bio text emphasizes with what the Postgres database actually records.

Once again, start by running the cell below with the `verbose` flag turned off; then run it again with the flag un-commented. Do the final results vary between runs? Does the agent behavior match what you expect? How is it different from the previous question?

In [ ]:
TUTORIAL_SYSTEM_PROMPT = (
    "You are a tutorial assistant with Olympic data tools (relational + document). "
    "Use tools for facts; do not invent athlete, edition, or country codes. "
    "Call tools yourself—do not ask the user to run queries. "
    "Keep tool use reasonable in depth (a few biography lookups per answer is enough). "
    "Prefer concise answers with citations to ids returned by tools."
)

NOTEBOOK_B_SPORTS = (
    "decathlon",
    "pole vault",
    "100 metres freestyle",
    "balance beam",
    "badminton",
)


def notebook_prompt_b() -> tuple[str, str]:
    sport = random.choice(NOTEBOOK_B_SPORTS)
    user = (
        f"Find athlete biography material tied to Olympic sport or event, using the theme "
        f"{sport!r} in a text-oriented search over the biography data. "
        f"Then cross-check that person against structured Olympic tables (starts, medals, editions) "
        f"and summarize: what the bio emphasizes versus what the relational data supports. "
        f"Cite ids where helpful."
    )
    return user, sport


prompt_b, sport_b = notebook_prompt_b()
print("Sport theme:", sport_b)

answer_b = chat_with_tools(
    settings,
    user_prompt=prompt_b,
    system_prompt=TUTORIAL_SYSTEM_PROMPT,
    #verbose=True
)
print(answer_b)

# II. Section 1

### Now that we have built our tool-calling loop and executed some example calls, let's practice using some of the other tools defined in our [toolbox](../toolbox/config/tools).

## Problem 1 — Athletes who competed for different countries across different Games

**Postgres-only flow:** The `country_represented` column in `athlete_events` records which country each athlete competed for at each edition. Some athletes changed nationality between Games — so their rows show different values of `country_represented` across editions. A sensible agent path is: get a list of medalists for a country, then call `pg_athlete_results` for each athlete with `edition_id_filter=0` (all editions) and check whether `country_represented` varies across their career.

The teaching goal is **chaining two tools together**: the first call returns a list of athletes, and the second call enriches each one with their full career history. Spotting the variation requires the agent to reason across multiple rows returned by the second tool — a step beyond simple lookup.

This problem is anchored to the **2012 London Summer Games** — write a plain string prompt, no f-strings needed.

The next cell holds your prompt and runs `chat_with_tools`.

**Docs:** same as the examples above: [Toolbox](https://mcp-toolbox.dev/documentation/introduction/), [OpenRouter](https://openrouter.ai/docs/quickstart), [function calling](https://platform.openai.com/docs/guides/function-calling).

In [ ]:
HOMEWORK_1_SYSTEM_PROMPT = (
    "You are a tutorial assistant with Olympic data tools (relational + document). "
    "Use tools for facts; do not invent athlete, edition, or country codes. "
    "Call tools yourself—do not ask the user to run queries. "
    "When checking career history, use pg_athlete_results with edition_id_filter=0 to get all editions. "
    "Only report athletes where country_represented actually differs across editions — ignore athletes "
    "who always competed for the same country. Show both countries for every athlete you report. "
    "Prefer concise answers with citations to ids returned by tools."
)

# TODO: Write a prompt asking which medalists from a country at the 2012 Olympics
# competed for a DIFFERENT country at some other point in their career.
# Hint: use pg_athlete_results (all editions) to check each athlete's full history.
# DEV TODO: REMOVE THIS LINE AND ANSWER KEY BELOW
prompt_1 = (
    "For the 2012 Summer Olympics, get the list of medalists who competed for France. "
    "For each athlete, fetch their full career results across all editions using athlete_id. "
    "Find any athlete who competed for a different country at some other Games — "
    "show their name, athlete_id, and both country codes (the other country and France). "
    "Only list athletes where the country actually changed; skip anyone who always competed for France."
)

answer_1 = chat_with_tools(
    settings,
    user_prompt=prompt_1,
    system_prompt=HOMEWORK_1_SYSTEM_PROMPT,
)
print(answer_1)


## Problem 2 — Highest gold medal winner in a random country

**Cross-database flow (Postgres first, then Mongo):** The question starts in **structured data**—you need counts of gold medals per athlete, which lives in Postgres. A sensible agent path is: resolve the country name to a NOC code, then call `pg_medal_events_for_country` across one or more editions to collect gold-medal rows, identify which athlete appears most often, and only then pull their biography from Mongo.

The teaching goal is **aggregation via the agent**: the existing tools return individual medal rows rather than pre-counted tallies, so the model must group and rank the results itself before reaching for the biography.

This cell picks **one random country** from a short list and builds a prompt that asks the agent to work through the full flow.

The next cell builds the user prompt and runs `chat_with_tools`.

**Docs:** same as the examples above: [Toolbox](https://mcp-toolbox.dev/documentation/introduction/), [OpenRouter](https://openrouter.ai/docs/quickstart), [function calling](https://platform.openai.com/docs/guides/function-calling).

In [ ]:
import random

HOMEWORK_2_SYSTEM_PROMPT = (
    "You are a tutorial assistant with Olympic data tools (relational + document). "
    "Use tools for facts; do not invent athlete, edition, or country codes. "
    "Call tools yourself—do not ask the user to run queries. "
    "When counting medals, tally the rows the tools return and rank them yourself. "
    "Only query the editions needed to answer the question — do not fetch every edition ever held. "
    "Prefer concise answers with citations to ids returned by tools."
)

HOMEWORK_2_COUNTRIES = (
    # TODO: Add in a few countries to get started! For example: "United States", "Kenya"
    # DEV TODO: REMOVE THIS LINE AND ANSWER KEY BELOW
    "United States",
    "Kenya",
    "Australia",
    "Cuba",
    "Hungary",
    "Norway",
)


def homework_prompt_2() -> tuple[str, str]:
    country = random.choice(HOMEWORK_2_COUNTRIES)
    # TODO: Write your prompt here! Use an f-string to insert the country name. Example: f"I live in {country}"
    # Hint: narrow the year range (e.g. 2000–2016) so the agent doesn't query every edition ever held.
    # DEV TODO: REMOVE THIS LINE AND ANSWER KEY BELOW
    user = (
        f"For the country {country!r}, find the athlete who won the most gold medals "
        f"at Summer Olympic Games between 2000 and 2016. "
        f"Resolve the country's NOC code, collect gold-medal rows for each edition in that range, "
        f"count golds per athlete, and identify the top winner. "
        f"Then fetch their biography from the document database and include a sentence or two. "
        f"Cite the athlete_id and gold medal count in your answer."
    )
    return user, country


prompt_2, country_2 = homework_prompt_2()
print("Country:", country_2)

answer_2 = chat_with_tools(
    settings,
    user_prompt=prompt_2,
    system_prompt=HOMEWORK_2_SYSTEM_PROMPT,
    max_rounds=8,
)
print(answer_2)


## Problem 3 — Best performing country in a sport over a range of years

**Cross-database flow (Postgres first, then Mongo):** Finding the best team over a span of Games requires looping across multiple editions. The agent should use `pg_list_games_by_year` to find every edition in the year range, call `pg_medal_events_for_country` with a sport filter for each edition to collect medal rows, tally the results across years, and then pull a Mongo biography for a standout athlete.

The teaching goal is **multi-variable prompting**: this problem has four moving parts that all need to land in the right place in your prompt — a randomly chosen country, a randomly chosen sport, and a `year_min`/`year_max` range that **you** set. Getting the agent to use all four correctly in one coherent query is the challenge.

This cell picks **one random country** and **one random sport**. You set `year_min` and `year_max` yourself, then write an f-string prompt that weaves all four variables together.

The next cell holds your variables and prompt, then runs `chat_with_tools`.

**Docs:** same as the examples above: [Toolbox](https://mcp-toolbox.dev/documentation/introduction/), [OpenRouter](https://openrouter.ai/docs/quickstart), [function calling](https://platform.openai.com/docs/guides/function-calling).

In [ ]:
import random

HOMEWORK_3_SYSTEM_PROMPT = (
    "You are a tutorial assistant with Olympic data tools (relational + document). "
    "Use tools for facts; do not invent athlete, edition, or country codes. "
    "Call tools yourself—do not ask the user to run queries. "
    "When spanning multiple Games, call medal tools once per edition and tally across them. "
    "Prefer concise answers with citations to ids returned by tools."
)

HOMEWORK_3_COUNTRIES = (
    # TODO: Add a few countries to pick from! For example: "United States", "China"
    # DEV TODO: REMOVE THIS LINE AND ANSWER KEY BELOW
    "United States",
    "China",
    "Germany",
    "Australia",
    "Jamaica",
)

HOMEWORK_3_SPORTS = (
    # TODO: Add a few Olympic sports to pick from! For example: "swimming", "athletics"
    # DEV TODO: REMOVE THIS LINE AND ANSWER KEY BELOW
    "swimming",
    "athletics",
    "gymnastics",
    "rowing",
    "cycling",
    "bobsledding"
)

# TODO: Set the year range you want to search across.
year_min = 2000  # replace with a year, e.g. 1992
year_max = 2012  # replace with a year, e.g. 2012


def homework_prompt_3() -> tuple[str, str, str]:
    country = random.choice(HOMEWORK_3_COUNTRIES)
    sport = random.choice(HOMEWORK_3_SPORTS)
    # TODO: Write your prompt here! Use an f-string to include country, sport, year_min, and year_max.
    # DEV TODO: REMOVE THIS LINE AND ANSWER KEY BELOW
    user = (
        f"For the country {country!r}, find all medal results in {sport!r} events "
        f"at Summer Olympic Games between {year_min} and {year_max}. "
        f"First look up the editions in that year range, then collect medal rows for each edition "
        f"filtered to {sport!r}, and tally the total medals across all those Games. "
        f"Identify the edition where {country!r} performed best in {sport!r}, then fetch a "
        f"biography for one of the top medalists from that edition. Cite athlete_ids."
    )
    return user, country, sport


prompt_3, country_3, sport_3 = homework_prompt_3()
print(f"Country: {country_3} | Sport: {sport_3} | Years: {year_min}-{year_max}")

answer_3 = chat_with_tools(
    settings,
    user_prompt=prompt_3,
    system_prompt=HOMEWORK_3_SYSTEM_PROMPT,
)
print(answer_3)


# II. Section 2

### Now we will implement new tools on our own, and use them to execute additional tool calls.

### II. Section 2.1 — Helper functions for registering new tools at runtime

Adding a new tool to MCP Toolbox requires three steps: writing a valid `kind: tool` YAML document to the config folder, registering the tool name in the right toolsets in `04_toolsets.yaml`, and restarting the Toolbox container so it re-reads the config.

The cell below automates all three steps while keeping the original config files intact:

- **`add_tool(...)`** writes the new tool to a scratch file (`99_student_tools.yaml`), backs up `04_toolsets.yaml` before patching it, restarts Toolbox, and waits until the service is ready before returning.
- **`remove_student_tools()`** deletes the scratch file, restores `04_toolsets.yaml` from the backup, and restarts Toolbox — leaving the repo exactly as it was.

The `_TOOL_TYPE_MAP` at the top maps the short type strings (`"postgres"`, `"mongo"`) used in student code to the full Toolbox YAML `type` and `source` values.

In [ ]:
import os
import re
import shutil
import subprocess
import textwrap
import time
import urllib.request
import urllib.error
from pathlib import Path

_TOOLS_DIR = repo_root() / "toolbox" / "config" / "tools"
_STUDENT_FILE = _TOOLS_DIR / "99_student_tools.yaml"
_TOOLSETS_FILE = _TOOLS_DIR / "04_toolsets.yaml"
_TOOLSETS_BACKUP = _TOOLS_DIR / "04_toolsets.yaml.bak"


_TOOL_TYPE_MAP = {
    "postgres": ("postgres-sql",       "olympics-postgres"),
    "mongo":    ("mongodb-aggregate",  "olympics-mongo"),
}


def _build_tool_yaml(tool_name, description, query, parameters, tool_type, collection=None):
    """Render a Toolbox tool YAML document as a string."""
    yaml_type, source = _TOOL_TYPE_MAP.get(tool_type, ("postgres-sql", "olympics-postgres"))
    lines = [
        "kind: tool",
        f"name: {tool_name}",
        f"type: {yaml_type}",
        f"source: {source}",
        "description: >",
        f"  {description.strip()}",
    ]
    if tool_type == "mongo":
        lines += [
            "database: ${MONGO_DB}",
            f"collection: {collection or 'olympic_athlete_biography'}",
            "readOnly: true",
        ]
        lines += ["pipelinePayload: |"]
        for line in textwrap.dedent(query).strip().splitlines():
            lines.append(f"  {line}")
        if parameters:
            lines.append("pipelineParams:")
            for p in parameters:
                lines += [
                    f"  - name: {p['name']}",
                    f"    type: {p.get('type', 'string')}",
                    f"    description: {p.get('description', '')}",
                ]
    else:
        if parameters:
            lines.append("parameters:")
            for p in parameters:
                lines += [
                    f"  - name: {p['name']}",
                    f"    type: {p.get('type', 'string')}",
                    f"    description: {p.get('description', '')}",
                ]
                if not p.get("required", True):
                    lines.append("    required: false")
        lines += ["statement: |"]
        for line in textwrap.dedent(query).strip().splitlines():
            lines.append(f"  {line}")
    lines += ["annotations:", "  readOnlyHint: true", "  destructiveHint: false"]
    return "\n".join(lines) + "\n"


def _append_to_toolsets(tool_name, toolsets):
    """Add tool_name to the tools list of each named toolset in 04_toolsets.yaml."""
    content = _TOOLSETS_FILE.read_text()
    for ts in toolsets:
        # Match the toolset block's tools list and append at the end of it.
        pattern = rf"(kind: toolset\nname: {re.escape(ts)}\ntools:(?:\n  - [^\n]+)*)"
        replacement = lambda m: m.group(0) + f"\n  - {tool_name}"
        content = re.sub(pattern, replacement, content)
    _TOOLSETS_FILE.write_text(content)


def _wait_for_toolbox(timeout: int = 30) -> None:
    """Poll the Toolbox health endpoint until it responds, or raise after timeout seconds."""
    url = settings.toolbox_base_url.rstrip("/") + "/"
    deadline = time.time() + timeout
    print(f"Waiting for Toolbox to be ready at {url} ...", end="", flush=True)
    while time.time() < deadline:
        try:
            urllib.request.urlopen(url, timeout=2)
            print(" ready.")
            return
        except Exception:
            print(".", end="", flush=True)
            time.sleep(1)
    raise TimeoutError(f"Toolbox did not become ready within {timeout}s.")


def _restart_toolbox() -> None:
    """Restart the toolbox container and wait until it is accepting connections."""
    result = subprocess.run(
        ["docker", "compose", "restart", "toolbox"],
        cwd=str(repo_root()),
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("docker compose restart failed:\n", result.stderr)
        return
    _wait_for_toolbox()


def add_tool(
    tool_name: str,
    description: str,
    sql: str,
    parameters: list[dict] | None = None,
    tool_type: str = "postgres",
    toolsets: list[str] | None = None,
    collection: str | None = None,
) -> None:
    """
    Write a new tool to 99_student_tools.yaml (never touches original files),
    register it in the specified toolsets, and restart Toolbox.

    Args:
        tool_name:   YAML tool name, e.g. "pg_medal_tally_by_edition".
        description: Description shown to the model.
        sql:         SQL statement (postgres) or JSON aggregation pipeline (mongo).
        parameters:  List of {"name", "type", "description"} dicts.
        tool_type:   "postgres" (default) or "mongo".
        toolsets:    Toolset names to register the tool in.
        collection:  MongoDB collection name (required when tool_type="mongo").
    """
    if tool_type not in _TOOL_TYPE_MAP:
        raise ValueError(f"tool_type must be one of {list(_TOOL_TYPE_MAP)}; got {tool_type!r}")
    if toolsets is None:
        toolsets = ["combined", "postgres_only"] if tool_type == "postgres" else ["combined", "mongo_only"]
    if parameters is None:
        parameters = []

    # Write tool YAML to the student scratch file (append if it already exists).
    yaml_doc = _build_tool_yaml(tool_name, description, sql, parameters, tool_type, collection)
    if _STUDENT_FILE.exists():
        _STUDENT_FILE.write_text(_STUDENT_FILE.read_text() + "\n---\n" + yaml_doc)
    else:
        _STUDENT_FILE.write_text(yaml_doc)
    print(f"Tool '{tool_name}' written to {_STUDENT_FILE.name}")

    # Back up toolsets file (once) then patch it.
    if not _TOOLSETS_BACKUP.exists():
        shutil.copy2(_TOOLSETS_FILE, _TOOLSETS_BACKUP)
        print(f"Toolsets backed up to {_TOOLSETS_BACKUP.name}")
    _append_to_toolsets(tool_name, toolsets)
    print(f"Registered '{tool_name}' in toolsets: {toolsets}")

    _restart_toolbox()
    print("Done. Call remove_student_tools() to undo all changes.")


def remove_student_tools() -> None:
    """
    Remove 99_student_tools.yaml and restore 04_toolsets.yaml from backup,
    then restart Toolbox. Leaves all original config files exactly as they were.
    """
    if _STUDENT_FILE.exists():
        _STUDENT_FILE.unlink()
        print(f"Removed {_STUDENT_FILE.name}")
    if _TOOLSETS_BACKUP.exists():
        shutil.copy2(_TOOLSETS_BACKUP, _TOOLSETS_FILE)
        _TOOLSETS_BACKUP.unlink()
        print("Restored 04_toolsets.yaml from backup")
    _restart_toolbox()
    print("Done. Config is back to its original state.")



## Problem 4 — Write the SQL for a medal tally tool

**Write your own SQL:** None of the existing tools can answer "which country topped the medal table at a given Games?" without calling `pg_medal_events_for_country` for every country separately — which is impractical. Your job is to write the SQL query that closes that gap for a new `pg_medal_tally_by_edition` tool.

**What the query should return:** One row per country with gold, silver, and bronze counts plus a total, sorted by gold descending. It takes a single parameter — `$1` — which will be the `edition_id` (an integer you can get from `pg_list_games_by_year`).

**The key SQL pattern:** `COUNT(DISTINCT result_id) FILTER (WHERE ...)` lets you pivot medal types into columns in a single aggregation pass. Using `DISTINCT result_id` is important — `athlete_events` has one row per athlete per event, so team events (relay, football, etc.) have many rows sharing the same `result_id`. Counting distinct `result_id` values gives one medal per event rather than one per athlete.

The next cell has the prompt and system prompt already written. Your only job is to fill in `medal_tally_sql` with a working query.

**Docs:** PostgreSQL [`COUNT` with `FILTER`](https://www.postgresql.org/docs/current/sql-expressions.html#SYNTAX-AGGREGATES); schema reference: `athlete_events(edition_id, country_represented, medal, result_id)`.

In [ ]:
# Tool metadata — already filled in for you.
tool_name = "pg_medal_tally_by_edition"
tool_description = (
    "Medal table for one Olympic edition: gold, silver, bronze, and total counts per country, "
    "sorted by gold descending. Pass the edition_id from pg_list_games_by_year."
)
tool_parameters = [
    {
        "name": "edition_id",
        "type": "integer",
        "description": "Olympic edition_id matching olympic_games.edition_id.",
    }
]
tool_type = "postgres"

# TODO: Write the SQL for the pg_medal_tally_by_edition tool.
# Requirements:
#   - Filter to edition_id = $1 (the single parameter this tool will accept)
#   - Count gold, silver, and bronze medals separately per country
#   - Include a total medal count column
#   - Group by country_represented, sort by gold DESC
# Hint: use COUNT(DISTINCT ae.result_id) FILTER (WHERE ae.medal = 'Gold') for each medal type
#       DISTINCT result_id avoids over-counting team sports (relay, football, etc.)
# DEV TODO: REMOVE THIS LINE AND ANSWER KEY BELOW
medal_tally_sql = """
SELECT
    ae.country_represented                                                        AS noc,
    COUNT(DISTINCT ae.result_id) FILTER (WHERE ae.medal = 'Gold')               AS gold,
    COUNT(DISTINCT ae.result_id) FILTER (WHERE ae.medal = 'Silver')             AS silver,
    COUNT(DISTINCT ae.result_id) FILTER (WHERE ae.medal = 'Bronze')             AS bronze,
    COUNT(DISTINCT ae.result_id) FILTER (WHERE ae.medal IS NOT NULL
                                          AND TRIM(ae.medal) <> '')             AS total
FROM athlete_events ae
WHERE ae.edition_id = $1
  AND ae.medal IS NOT NULL
  AND TRIM(ae.medal) <> ''
GROUP BY ae.country_represented
ORDER BY gold DESC, silver DESC, bronze DESC
LIMIT 50
"""

# Step 2: write the tool to 99_student_tools.yaml, patch the toolsets, and restart Toolbox.
add_tool(tool_name, tool_description, medal_tally_sql, tool_parameters, tool_type)

# Step 3: test the new tool with the agent.
HOMEWORK_4_SYSTEM_PROMPT = (
    "You are a tutorial assistant with Olympic data tools (relational + document). "
    "Use tools for facts; do not invent athlete, edition, or country codes. "
    "Call tools yourself—do not ask the user to run queries. "
    "Use pg_list_games_by_year to resolve a year to an edition_id, then use "
    "pg_medal_tally_by_edition to get the full medal table for that edition. "
    "Prefer concise answers with citations to ids returned by tools."
)

prompt_4 = (
    "Use the available tools to find the edition_id for the 2008 Summer Olympics, "
    "then call pg_medal_tally_by_edition with that edition_id to get the full medal table. "
    "Which country finished first, second, and third in gold medals? "
    "Cite the edition_id and the NOC codes from the tool output."
)

answer_4 = chat_with_tools(
    settings,
    user_prompt=prompt_4,
    system_prompt=HOMEWORK_4_SYSTEM_PROMPT,
    #verbose=True
)
print(answer_4)

## Problem 5 — Write a MongoDB pipeline for country + keyword biography search

**Write your own pipeline:** The existing Mongo tools search biographies by country (`mongo_biographies_by_country`) or by keyword (`mongo_biography_text_search`) — but not both at once. Your job is to write a new `mongo_bios_by_country_and_text` tool that narrows results to a specific NOC **and** a keyword in the same query.

**What the pipeline should do:** Accept two parameters — `country_noc` (e.g. `"KEN"`) and `text_fragment` (e.g. `"marathon"`) — and return a random sample of up to 10 biography documents where both conditions are true: the athlete's `country_noc` matches exactly, and either `description` or `special_notes` contains the keyword (case-insensitive).

**The key MongoDB pattern:** A `$match` stage with `$and` lets you combine multiple conditions. Inside `$and`, use an exact match for `country_noc` and a `$or` with `$regex` for the text search — then pipe the matches into `$sample` for a random subset.

Parameters are injected into the pipeline with `{{json .param_name}}` (Toolbox's template syntax, shown in the existing Mongo tools in `03_mongo_tools.yaml`).

The next cell has the tool metadata and prompt already written. Your only job is to fill in `bios_pipeline` with a working aggregation pipeline.

**Docs:** [MongoDB `$match`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/match/), [`$and`](https://www.mongodb.com/docs/manual/reference/operator/query/and/), [`$regex`](https://www.mongodb.com/docs/manual/reference/operator/query/regex/), [`$sample`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/sample/); [Toolbox pipeline parameters](https://mcp-toolbox.dev/documentation/configure.md).

In [ ]:
# Tool metadata — already filled in for you.
tool_name_5 = "mongo_bios_by_country_and_text"
tool_description_5 = (
    "Search athlete biography documents for a keyword, filtered to a specific country NOC. "
    "Returns a random sample of up to 10 matching documents. "
    "Use when you need biographies from a specific country that mention a particular theme."
)
tool_parameters_5 = [
    {
        "name": "country_noc",
        "type": "string",
        "description": "National Olympic Committee code to filter by (e.g. KEN, FRA).",
    },
    {
        "name": "text_fragment",
        "type": "string",
        "description": "Keyword or phrase to search for in description and special_notes (case-insensitive).",
    },
]
tool_type_5 = "mongo"
collection_5 = "olympic_athlete_biography"

# TODO: Write the MongoDB aggregation pipeline for mongo_bios_by_country_and_text.
# Requirements:
#   - $match stage using $and to combine:
#       * exact match on country_noc using {{json .country_noc}}
#       * $or regex search on description and special_notes using {{json .text_fragment}}
#   - $sample stage returning 10 documents
# Hint: look at mongo_biography_text_search in toolbox/config/tools/03_mongo_tools.yaml
#       for the $regex + {{json ...}} template syntax.
# DEV TODO: REMOVE THIS LINE AND ANSWER KEY BELOW
bios_pipeline = """
[
  {
    "$match": {
      "$and": [
        { "country_noc": {{json .country_noc}} },
        {
          "$or": [
            { "description":    { "$regex": {{json .text_fragment}}, "$options": "i" } },
            { "special_notes":  { "$regex": {{json .text_fragment}}, "$options": "i" } }
          ]
        }
      ]
    }
  },
  { "$sample": { "size": 10 } }
]
"""

# Step 2: register the tool and restart Toolbox.
add_tool(
    tool_name_5,
    tool_description_5,
    bios_pipeline,
    tool_parameters_5,
    tool_type_5,
    collection=collection_5,
)

# Step 3: test the new tool with the agent.
HOMEWORK_5_SYSTEM_PROMPT = (
    "You are a tutorial assistant with Olympic data tools (relational + document). "
    "Use tools for facts; do not invent athlete, edition, or country codes. "
    "Call tools yourself—do not ask the user to run queries. "
    "Use mongo_bios_by_country_and_text to find biographies from a specific country "
    "that match a keyword, then cross-check any athlete_ids with Postgres if helpful. "
    "Prefer concise answers with citations to ids returned by tools."
)

country = "Kenya"
keyword = "marathon"

prompt_5 = (
    f"Use mongo_bios_by_country_and_text to find athletes from {country} with biographies "
    f"that mention '{keyword}'. Summarize what you find — who are the athletes, and what do "
    f"their biographies say about {keyword}? Cite athlete_ids."
)

answer_5 = chat_with_tools(
    settings,
    user_prompt=prompt_5,
    system_prompt=HOMEWORK_5_SYSTEM_PROMPT,
    #verbose=True
)
print(answer_5)


### II. Section 2.2 — Clean up when you are done

Run the cell below to undo all changes made in this section. `remove_student_tools()` will:

- Delete `99_student_tools.yaml` (removes all tools you registered)
- Restore `04_toolsets.yaml` from the backup taken before your first `add_tool()` call
- Restart Toolbox and wait until it is ready

After this cell runs, the config is back to exactly the state it was in before Problem 4.

In [ ]:
remove_student_tools()